# Turbulent box tutorial

A periodic box of isothermal gas stirred by a large-scale forcing field is the standard numerical
laboratory for supersonic turbulence (the interstellar medium, molecular clouds, star formation).
In this tutorial we run RAMSES' turbulence-driving module as a small **numerical experiment**:
we compare **decaying** vs **driven** turbulence, measure how the statistics evolve in time, look
at the density structure and its PDF, and make a movie.

## 1. Install the tools
We use **RAMSES** (with its turbulence module) and **osyris** for the analysis.

In [ ]:
%pip install osyris

Download RAMSES:

In [ ]:
%%bash
git clone https://github.com/ramses-organisation/ramses.git

The turbulence module needs a Fortran compiler, MPI, and the **FFTW** library
(see `ramses/turb/README`). Compile RAMSES in 3D with the turbulence module on
(`NDIM=3 USE_TURB=1 MPI=1`). This builds the executable `ramses/bin/ramses3d`:

In [ ]:
%%bash
cd ramses/bin
make clean
make NDIM=3 USE_TURB=1 MPI=1
ls -l ramses3d

## 2. The experiment: decaying vs driven turbulence
We compare three runs in the same $64^3$ periodic box, provided as namelists next to this notebook:

| run | driving | what we expect |
|---|---|---|
| **`decay`** | none &mdash; start from a turbulent field | turbulence **decays** freely |
| **`driven_static`** | continuous, one *frozen* forcing pattern | rises to a **steady state** |
| **`driven_evolving`** | continuous, *time-evolving* (OU) forcing | steady state, time-varying pattern |

The key switches are in `&TURB_PARAMS`: `driven_turb` (drive or not), `turb_evolving` (frozen vs
time-correlated forcing) and `initial_turb` (seed an initial field). We write frequent snapshots
(`foutput`) so we can follow the statistics in time, and `driven_static` also writes **movie frames**
(`&MOVIE_PARAMS`) that we use at the end.

Set the path to the executable you just built and run the three experiments &mdash; this is the main
computation, done explicitly so you can see and control it (each is a cheap $64^3$ run, a few
minutes total):

In [ ]:
import os, glob, subprocess
import numpy as np
import osyris
import matplotlib.pyplot as plt

ramses_exe_full_path = os.path.abspath("ramses/bin/ramses3d")   # <-- adapt if built elsewhere
MPIRUN = "mpirun -np 4"                                         # or "" for serial

for case in ["decay", "driven_static", "driven_evolving"]:
    os.makedirs("runs/" + case, exist_ok=True)
    subprocess.run("cp %s.nml runs/%s/ && cd runs/%s && %s %s %s.nml > run.log 2>&1"
                   % (case, case, case, MPIRUN, ramses_exe_full_path, case), shell=True)
    n = len(glob.glob("runs/%s/output_*" % case))
    print("%-16s %d snapshots" % (case, n))

## 3. Visualise the velocity field
A density slice of the driven, static run at steady state with the velocity field overlaid as
streamlines &mdash; the turbulent eddies, dense filaments and voids:

![velocity streamlines](turb_velocity.png)


In [ ]:
def last_output(path):
    return max(int(os.path.basename(p).split('_')[1]) for p in glob.glob(path + "/output_*"))

data = osyris.RamsesDataset(last_output("runs/driven_static"), path="runs/driven_static").load()
osyris.map(data["mesh"].layer("density", norm="log"),
           data["mesh"].layer("velocity", mode="stream", color="w"),
           direction="z")

## 4. Turbulence statistics in time
For each snapshot we measure the **RMS Mach number** $\mathcal{M}=v_\mathrm{rms}/c_s$, the **density
dispersion** $\sigma_s=\mathrm{std}(\ln\rho/\langle\rho\rangle)$, and the **kinetic energy**.

In [ ]:
def turb_stats(run):
    snaps = sorted(int(os.path.basename(p).split('_')[1]) for p in glob.glob(run + "/output_*"))
    t, mach, sig, ek = [], [], [], []
    for s in snaps:
        d = osyris.RamsesDataset(s, path=run).load()
        rho = d["mesh"]["density"].values
        v   = d["mesh"]["velocity"]; v2 = v.x.values**2 + v.y.values**2 + v.z.values**2
        cs  = np.sqrt(d["mesh"]["pressure"].values / rho)
        t.append(float(d.meta["time"].magnitude) / d.meta["unit_t"])
        mach.append(np.sqrt(np.mean(v2)) / np.mean(cs))
        sig.append(np.std(np.log(rho / np.mean(rho))))
        ek.append(0.5 * np.mean(rho * v2))
    return map(np.array, (t, mach, sig, ek))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for name, col in [("decay","C0"), ("driven_static","C1"), ("driven_evolving","C2")]:
    t, mach, sig, ek = turb_stats("runs/" + name)
    ax[0].plot(t, mach, col, marker=".", ms=3, label=name)
    ax[1].plot(t, sig,  col, marker=".", ms=3)
    ax[2].plot(t, ek / ek[ek > 0][0], col, marker=".", ms=3)
ax[0].set(xlabel="t [code]", ylabel=r"$\mathcal{M}$", title="RMS Mach number"); ax[0].legend()
ax[1].set(xlabel="t [code]", ylabel=r"$\sigma_s$", title="density dispersion")
ax[2].set(xlabel="t [code]", ylabel=r"$E_k / E_{k,0}$", title="kinetic energy", yscale="log")
fig.tight_layout()

- **`decay`**: a brief settling transient, then $\mathcal{M}$ and $E_k$ **decay** &mdash; with no
  energy input the turbulence dies away.
- **`driven_static`** / **`driven_evolving`**: $\mathcal{M}$, $\sigma_s$ and $E_k$ **rise and saturate**
  &mdash; forcing balances dissipation at a statistical steady state.

(These parameters drive *very* supersonic turbulence, $\mathcal{M}\sim15$&ndash;20; lower `turb_rms`
for a gentler $\mathcal{M}\sim5$.)

![turbulence statistics vs time](turb_stats.png)


## 5. The density PDF
Supersonic isothermal turbulence has a **log-normal** density PDF: $s=\ln(\rho/\langle\rho\rangle)$
is nearly Gaussian, with width set by the Mach number,
$\sigma_s^2\approx\ln(1+b^2\mathcal{M}^2)$, where $b$ depends on the forcing mix.

![density PDF](turb_pdf.png)


In [ ]:
run = "runs/driven_static"
rho = osyris.RamsesDataset(last_output(run), path=run).load()["mesh"]["density"].values
s = np.log(rho / np.mean(rho)); sig = np.std(s)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(s, bins=60, density=True, alpha=0.6, label="simulation")
x = np.linspace(s.min(), s.max(), 200)
ax.plot(x, np.exp(-(x + sig**2/2)**2 / (2*sig**2)) / np.sqrt(2*np.pi*sig**2),
        "r-", lw=2, label=r"log-normal, $\sigma_s=%.2f$" % sig)
ax.set(xlabel=r"$s=\ln(\rho/\langle\rho\rangle)$", ylabel="PDF",
       title="Density PDF, driven turbulence (steady state)"); ax.legend()

## 6. Solenoidal vs compressive forcing
The *type* of stirring matters as much as its strength. We re-run the driven case with purely
**solenoidal** forcing (`comp_frac=0.0`) and purely **compressive** forcing (`comp_frac=1.0`):

In [ ]:
for cf, tag in [("0.0", "comp_sol"), ("1.0", "comp_comp")]:
    os.makedirs("runs/" + tag, exist_ok=True)
    conf = open("driven_static.nml").read().replace("comp_frac=0.5", "comp_frac=" + cf)
    open("runs/%s/%s.nml" % (tag, tag), "w").write(conf)
    subprocess.run("cd runs/%s && %s %s %s.nml > run.log 2>&1"
                   % (tag, MPIRUN, ramses_exe_full_path, tag), shell=True)
    print(tag, "done")

fig, ax = plt.subplots(figsize=(6, 4))
for tag, lab, col in [("comp_sol","solenoidal (comp_frac=0)","C0"),
                      ("comp_comp","compressive (comp_frac=1)","C3")]:
    d = osyris.RamsesDataset(last_output("runs/"+tag), path="runs/"+tag).load()
    rho = d["mesh"]["density"].values; s = np.log(rho/np.mean(rho))
    v = d["mesh"]["velocity"]; v2 = v.x.values**2+v.y.values**2+v.z.values**2
    M = np.sqrt(np.mean(v2))/np.mean(np.sqrt(d["mesh"]["pressure"].values/rho))
    ax.hist(s, bins=60, density=True, alpha=0.5, color=col,
            label=r"%s: $\sigma_s$=%.2f, M=%.0f" % (lab, np.std(s), M))
ax.set(xlabel=r"$s=\ln(\rho/\langle\rho\rangle)$", ylabel="PDF",
       title="Solenoidal vs compressive forcing"); ax.legend()

**Compressive forcing produces a much broader density PDF** than solenoidal forcing at the
same Mach number ($b\simeq1$ vs $b\simeq1/3$) &mdash; a key knob in star-formation turbulence studies.

![solenoidal vs compressive](turb_comp_sol.png)


## 7. A movie of the turbulence developing
`driven_static.nml` contains a `&MOVIE_PARAMS` block, so RAMSES wrote **movie frames** directly
during the run (`runs/driven_static/movie1/dens_*.map`). As in the dense-core-collapse tutorial, we
read those binary frames and stitch them into an mp4 with `ffmpeg` &mdash; no need to reprocess the
full snapshots.

In [ ]:
from scipy.io import FortranFile
from matplotlib.colors import LogNorm

mp = "runs/driven_static/movie1"
frames = sorted(int(os.path.basename(f)[5:10]) for f in glob.glob(mp + "/dens_*.map"))
os.makedirs("turb_frames", exist_ok=True)
for n, i in enumerate(frames):
    f = FortranFile("%s/dens_%05d.map" % (mp, i))
    f.read_reals("d"); nx, ny = f.read_ints()                 # header: time, box; then frame size
    img = np.array(f.read_reals("f4"), float).reshape(nx, ny) # the density frame
    fig, a = plt.subplots(figsize=(5, 5))
    a.imshow(img, origin="lower", norm=LogNorm(0.3, 20), cmap="inferno"); a.set_axis_off()
    fig.savefig("turb_frames/f_%04d.png" % n, dpi=90, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

subprocess.call("ffmpeg -y -framerate 12 -i turb_frames/f_%04d.png -pix_fmt yuv420p turb_movie.mp4",
                shell=True)
print("wrote turb_movie.mp4 from %d RAMSES movie frames" % len(frames))

A few frames: the gas is near-uniform at first, then develops the familiar network of
shocks, dense filaments and voids as the forcing spins up the turbulence.

![density frames over time](turb_movie_montage.png)

## Going further
- **Forcing spectrum.** Try `forcing_power_spectrum='parabolic'` (see `tests/turb/driving-parabolic`)
  or `'konstandin'`.
- **Resolution.** Raise `levelmin=levelmax` to 7 ($128^3$) and watch the small-scale structure sharpen.
- **Power spectrum.** Compute the velocity power spectrum and look for the expected slope
  (Kolmogorov $-5/3$ subsonic, closer to Burgers $-2$ when strongly supersonic).